In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import os
import glob
import re
import pandas as pd
import seaborn as sns
from scipy.interpolate import griddata
import matplotlib.gridspec as gridspec
import time as time_module
import shutil

# ================= 配置参数 =================
RESULTS_ROOT = "./pinn_results"  # 结果根目录

# ================= 辅助函数 =================
def find_results_folders():
    """查找所有PINN和MC结果文件夹"""
    # 获取所有目录
    all_dirs = [d for d in sorted(glob.glob(os.path.join(RESULTS_ROOT, "*")), key=os.path.getmtime, reverse=True)
                if os.path.isdir(d)]
    
    print("\n所有找到的结果文件夹:")
    for d in all_dirs:
        print(f" - {os.path.basename(d)}")
    
    # 分类文件夹
    pinn_dirs = []
    mc_dirs = []
    
    for d in all_dirs:
        folder_name = os.path.basename(d).lower()
        
        # 检查是否是PINN结果
        if any(keyword in folder_name for keyword in ['pinn', 'pi100', 'nn']):
            pinn_dirs.append(d)
        
        # 检查是否是MC结果
        if any(keyword in folder_name for keyword in ['mc', 'monte', 'sim', 'pi100']):
            mc_dirs.append(d)
    
    # 如果没有找到特定文件夹，使用所有目录
    if not pinn_dirs:
        print("警告: 未找到PINN结果文件夹，使用所有文件夹作为备选")
        pinn_dirs = all_dirs.copy()
    
    if not mc_dirs:
        print("警告: 未找到MC结果文件夹，使用所有文件夹作为备选")
        mc_dirs = all_dirs.copy()
    
    # 获取最新匹配的文件夹
    latest_pinn = pinn_dirs[0] if pinn_dirs else None
    latest_mc = mc_dirs[0] if mc_dirs else None
    
    print(f"\n选择的PINN文件夹: {os.path.basename(latest_pinn) if latest_pinn else '无'}")
    print(f"选择的MC文件夹: {os.path.basename(latest_mc) if latest_mc else '无'}")
    
    return latest_pinn, latest_mc

def load_results(dir_path):
    """从文件夹加载结果和参数"""
    # 尝试加载结果文件
    result_files = []
    for f in os.listdir(dir_path):
        if f.endswith(("_results.npy", "results.npz", "pinn_results.npy", "mc_results.npy")):
            result_files.append(f)
    
    if not result_files:
        available_files = ', '.join(os.listdir(dir_path))
        raise FileNotFoundError(f"在 {dir_path} 中未找到结果文件。可用文件: {available_files}")
    
    # 加载第一个找到的结果文件
    results_path = os.path.join(dir_path, result_files[0])
    print(f"从 {results_path} 加载结果数据")
    
    # 根据扩展名加载
    if results_path.endswith('.npz'):
        data = np.load(results_path)
        # 尝试查找包含点数据的键
        keys = [k for k in data.files if 'pinn' in k.lower() or 'mc' in k.lower() or 'result' in k.lower()]
        if not keys:
            keys = data.files
        results = data[keys[0]]
    else:
        results = np.load(results_path)
    
    # 加载参数
    params_path = os.path.join(dir_path, "simulation_parameters.txt")
    params = {}
    if os.path.exists(params_path):
        print(f"从 {params_path} 加载参数")
        with open(params_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                # 跳过空行和分隔符
                if not line.strip() or line.startswith('---') or line.startswith('==='):
                    continue
                
                # 尝试不同格式
                separators = [":", "=", "-", " "]
                for sep in separators:
                    if sep in line:
                        parts = line.split(sep, 1)
                        if len(parts) == 2:
                            key = parts[0].strip().lower().replace(' ', '_')
                            value = parts[1].strip()
                            
                            # 尝试转换数值
                            try:
                                if '.' in value:
                                    params[key] = float(value)
                                else:
                                    params[key] = int(value)
                            except ValueError:
                                params[key] = value
                            break
    
    return results, params

def create_comparison_grid(A_pinn, t_pinn, C_pinn, A_mc, t_mc, C_mc):
    """创建比较网格，将MC结果插值到PINN网格上"""
    # 确定公共网格范围
    A_min = max(A_pinn.min(), A_mc.min())
    A_max = min(A_pinn.max(), A_mc.max())
    t_min = max(t_pinn.min(), t_mc.min())
    t_max = min(t_pinn.max(), t_mc.max())
    
    print(f"\n创建比较网格: A范围[{A_min:.2f}, {A_max:.2f}], t范围[{t_min:.2f}, {t_max:.2f}]")
    
    # 使用更密集的网格进行更准确比较
    n_points = min(200, len(np.unique(A_pinn)), len(np.unique(A_mc)))
    A_grid = np.linspace(A_min, A_max, n_points)
    t_grid = np.linspace(t_min, t_max, n_points)
    A_mesh, t_mesh = np.meshgrid(A_grid, t_grid, indexing='ij')
    
    print(f"网格尺寸: {A_mesh.shape}")
    
    # 将PINN结果插值到网格
    print("插值PINN结果...")
    C_pinn_grid = griddata(
        (A_pinn, t_pinn), C_pinn, 
        (A_mesh, t_mesh), method='cubic'
    )
    
    # 将MC结果插值到网格
    print("插值MC结果...")
    C_mc_grid = griddata(
        (A_mc, t_mc), C_mc, 
        (A_mesh, t_mesh), method='linear'  # 对MC数据使用线性插值
    )
    
    # 计算绝对误差和相对误差
    print("计算误差...")
    absolute_error = np.abs(C_pinn_grid - C_mc_grid)
    relative_error = absolute_error / (np.abs(C_mc_grid) + 1e-6)  # 避免除以零
    
    return A_mesh, t_mesh, C_pinn_grid, C_mc_grid, absolute_error, relative_error
def plot_3d_comparison(A, t, C_pinn, C_mc, diff, title, save_path):
    """Draw 3D comparison plots (all labels in English)"""
    print(f"Saving 3D comparison plot to {save_path}")
    fig = plt.figure(figsize=(18, 12))
    
    # Create 3D plots
    ax1 = fig.add_subplot(221, projection='3d')
    surf1 = ax1.plot_surface(A, t, C_pinn, cmap='viridis', alpha=0.8, rstride=1, cstride=1)
    ax1.set_title('PINN Solution')
    ax1.set_xlabel('Asset Price (A)')
    ax1.set_ylabel('Time (t)')
    ax1.set_zlabel('Option Price')
    fig.colorbar(surf1, ax=ax1, shrink=0.5, label='Price')
    
    ax2 = fig.add_subplot(222, projection='3d')
    surf2 = ax2.plot_surface(A, t, C_mc, cmap='plasma', alpha=0.8, rstride=1, cstride=1)
    ax2.set_title('Monte Carlo Simulation')
    ax2.set_xlabel('Asset Price (A)')
    ax2.set_ylabel('Time (t)')
    ax2.set_zlabel('Option Price')
    fig.colorbar(surf2, ax=ax2, shrink=0.5, label='Price')
    
    ax3 = fig.add_subplot(223, projection='3d')
    surf3 = ax3.plot_surface(A, t, diff, cmap='coolwarm', alpha=0.8, rstride=1, cstride=1)
    ax3.set_title('Absolute Error')
    ax3.set_xlabel('Asset Price (A)')
    ax3.set_ylabel('Time (t)')
    ax3.set_zlabel('|PINN - MC|')
    fig.colorbar(surf3, ax=ax3, shrink=0.5, label='Error')
    
    ax4 = fig.add_subplot(224, projection='3d')
    log_diff = np.log10(np.clip(diff, 1e-8, None))
    surf4 = ax4.plot_surface(A, t, log_diff, cmap='coolwarm', alpha=0.8, rstride=1, cstride=1)
    ax4.set_title('Log Absolute Error')
    ax4.set_xlabel('Asset Price (A)')
    ax4.set_ylabel('Time (t)')
    ax4.set_zlabel('log10(|PINN - MC|)')
    fig.colorbar(surf4, ax=ax4, shrink=0.5, label='log10(Error)')
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close(fig)
    return save_path

def plot_time_series_comparison(A_pinn, t_pinn, C_pinn, A_mc, t_mc, C_mc, params, save_path):
    """Compare PINN and MC results at specific time points (all labels in English)"""
    print(f"Saving time series comparison plot to {save_path}")
    # Get expiration time
    tau_keys = ['tau', 't_expiry', 'option_expiry', 'expiry_time']
    tau = params.get(next((k for k in tau_keys if k in params), 'tau'), 
                   max(t_pinn.max(), t_mc.max()))
    
    # Select meaningful time points (0, τ/2, τ)
    t_values = [0, tau/2, tau]
    labels = [f"Start (t=0)", f"Mid-term (t={tau/2:.2f})", f"Expiry (t={tau:.2f})"]
    colors = ['blue', 'green', 'red']
    linestyles = ['-', '--', ':']
    
    plt.figure(figsize=(12, 7))
    
    for i, t_val in enumerate(t_values):
        # PINN data points
        t_tolerance = max(0.01, tau * 0.05)
        t_mask_pinn = (np.abs(t_pinn - t_val) <= t_tolerance)
        
        if not np.any(t_mask_pinn):
            closest_idx = np.argmin(np.abs(t_pinn - t_val))
            t_val = t_pinn[closest_idx]
            print(f"Using closest PINN time point: t={t_val:.4f} (original {t_values[i]:.4f})")
            t_mask_pinn = (np.abs(t_pinn - t_val) <= 1e-3)
        
        A_t_pinn = A_pinn[t_mask_pinn]
        C_t_pinn = C_pinn[t_mask_pinn]
        
        # Sort by A
        sort_idx = np.argsort(A_t_pinn)
        plt.plot(A_t_pinn[sort_idx], C_t_pinn[sort_idx], 
                 color=colors[i], linestyle=linestyles[i], linewidth=2, 
                 label=f"PINN - {labels[i]}")
        
        # MC data points
        t_mask_mc = (np.abs(t_mc - t_val) <= t_tolerance)
        
        if not np.any(t_mask_mc):
            closest_idx = np.argmin(np.abs(t_mc - t_val))
            t_val = t_mc[closest_idx]
            print(f"Using closest MC time point: t={t_val:.4f} (original {t_values[i]:.4f})")
            t_mask_mc = (np.abs(t_mc - t_val) <= 1e-3)
        
        A_t_mc = A_mc[t_mask_mc]
        C_t_mc = C_mc[t_mask_mc]
        
        # Sort by A
        sort_idx = np.argsort(A_t_mc)
        plt.scatter(A_t_mc[sort_idx], C_t_mc[sort_idx], 
                    color=colors[i], marker='o', s=40, alpha=0.7,
                    label=f"MC - {labels[i]}")
    
    # Get strike price
    strike_keys = ['k', 'strike_price', 'option_strike']
    K = params.get(next((k for k in strike_keys if k in params), 'strike_price'), 50.0)
    
    # Add strike price reference line
    plt.axvline(x=K, color='gray', linestyle='-.', alpha=0.5, 
                label=f'Strike Price (K={K:.1f})')
    
    # Set labels
    plt.xlabel("Asset Price (A)")
    plt.ylabel("Option Price (C)")
    
    # Set title
    pi_val = params.get('pi', 'Unknown')
    beta_val = params.get('beta', 'Unknown')
    alpha_val = params.get('alpha', 'Unknown')
    
    plt.title(f"Comparison of PINN and MC at Different Time Points\nπ={pi_val}, β={beta_val}, α={alpha_val}, τ={tau:.2f}, K={K:.1f}")
    plt.legend(loc='best', frameon=True, framealpha=0.8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    plt.savefig(save_path, dpi=150)
    plt.close()
    return save_path

def plot_error_heatmaps(A, t, abs_error, rel_error, params, save_path):
    """Draw error heatmaps (all labels in English)"""
    print(f"Saving error heatmaps to {save_path}")
    fig = plt.figure(figsize=(18, 8))
    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1])
    
    # Get strike price
    strike_keys = ['k', 'strike_price', 'option_strike']
    K = params.get(next((k for k in strike_keys if k in params), 'strike_price'), 50.0)
    
    # Absolute error heatmap
    ax1 = plt.subplot(gs[0])
    im1 = ax1.imshow(abs_error.T, aspect='auto', 
                    extent=[A.min(), A.max(), t.max(), t.min()],
                    cmap='coolwarm', 
                    origin='lower')
    
    # Mark strike price
    K_pos = (K - A.min()) / (A.max() - A.min()) * (len(A) - 1)
    ax1.axvline(x=K_pos, color='purple', linestyle='--', alpha=0.7, label=f'Strike Price K={K:.1f}')
    
    ax1.set_title('Absolute Error Heatmap (|PINN - MC|)')
    ax1.set_xlabel('Asset Price (A)')
    ax1.set_ylabel('Time (t)')
    ax1.legend(loc='upper right')
    cbar1 = fig.colorbar(im1, ax=ax1, shrink=0.7)
    cbar1.set_label('Error Magnitude')
    
    # Relative error heatmap
    ax2 = plt.subplot(gs[1])
    # Cap relative error for better visualization
    rel_error_capped = np.clip(rel_error, 0, 2)
    im2 = ax2.imshow(rel_error_capped.T, aspect='auto', 
                    extent=[A.min(), A.max(), t.max(), t.min()],
                    cmap='coolwarm', 
                    origin='lower', vmax=2.0)
    
    # Mark strike price
    ax2.axvline(x=K_pos, color='purple', linestyle='--', alpha=0.7, label=f'Strike Price K={K:.1f}')
    
    ax2.set_title('Relative Error Heatmap (|PINN - MC|/|MC|)')
    ax2.set_xlabel('Asset Price (A)')
    ax2.set_ylabel('Time (t)')
    ax2.legend(loc='upper right')
    cbar2 = fig.colorbar(im2, ax=ax2, shrink=0.7)
    cbar2.set_label('Relative Error')
    
    # Add main title
    plt.suptitle(f'Error Analysis (τ={t.max():.2f}, K={K:.1f})', fontsize=16)
    plt.tight_layout()
    
    plt.savefig(save_path, dpi=150)
    plt.close(fig)
    return save_path

def analyze_error_statistics(abs_error, rel_error, save_path):
    """Analyze error statistics and plot distributions (all labels in English)"""
    print(f"Saving error distribution plots to {save_path}")
    # Calculate basic statistics
    abs_err_flat = abs_error.flatten()
    rel_err_flat = rel_error.flatten()
    
    error_stats = {
        "mean_absolute_error": np.nanmean(abs_err_flat),
        "max_absolute_error": np.nanmax(abs_err_flat),
        "median_absolute_error": np.nanmedian(abs_err_flat),
        "rmse": np.sqrt(np.nanmean(np.square(abs_err_flat))),
        "mean_relative_error": np.nanmean(rel_err_flat) * 100,
        "median_relative_error": np.nanmedian(rel_err_flat) * 100,
        "std_absolute_error": np.nanstd(abs_err_flat),
        "min_absolute_error": np.nanmin(abs_err_flat),
        "min_relative_error": np.nanmin(rel_err_flat) * 100,
        "max_relative_error": np.nanmax(rel_err_flat) * 100
    }
    
    # Print statistics
    print("\nError Statistics:")
    for stat, value in error_stats.items():
        print(f"{stat}: {value:.6f}")
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    
    # Absolute error distribution
    ax1.hist(abs_err_flat, bins=50, color='purple', alpha=0.7)
    ax1.set_title('Absolute Error Distribution')
    ax1.set_xlabel('|PINN - MC|')
    ax1.set_ylabel('Frequency')
    ax1.grid(True, alpha=0.3)
    
    # Add statistical markers
    ax1.axvline(error_stats["median_absolute_error"], color='r', linestyle='--', 
                label=f'Median: {error_stats["median_absolute_error"]:.4f}')
    ax1.axvline(error_stats["mean_absolute_error"], color='g', linestyle='--', 
                label=f'Mean: {error_stats["mean_absolute_error"]:.4f}')
    ax1.legend()
    
    # Relative error distribution
    # Filter extreme values
    rel_err_filt = rel_err_flat[(rel_err_flat > np.percentile(rel_err_flat, 1)) & 
                               (rel_err_flat < np.percentile(rel_err_flat, 99))]
    ax2.hist(rel_err_filt * 100, bins=50, color='teal', alpha=0.7)
    ax2.set_title('Relative Error Distribution (1-99 Percentile)')
    ax2.set_xlabel('Relative Error (%)')
    ax2.set_ylabel('Frequency')
    ax2.grid(True, alpha=0.3)
    
    # Add statistical markers
    ax2.axvline(np.nanmedian(rel_err_filt)*100, color='r', linestyle='--', 
                label=f'Median: {np.nanmedian(rel_err_filt)*100:.2f}%')
    ax2.axvline(np.nanmean(rel_err_filt)*100, color='g', linestyle='--', 
                label=f'Mean: {np.nanmean(rel_err_filt)*100:.2f}%')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    
    return error_stats, save_path

def create_comparison_report(pinn_dir, mc_dir, comp_dir, params, error_stats):
    """创建比较报告"""
    report_path = os.path.join(comp_dir, "comparison_report.txt")
    print(f"创建比较报告: {report_path}")
    
    # 提取参数
    pi_val = params.get('pi', '未知')
    beta_val = params.get('beta', '未知')
    alpha_val = params.get('alpha', '未知')
    r_val = params.get('r', '未知')
    T_val = params.get('t', '未知')
    tau_val = params.get('tau', '未知')
    K_val = params.get('k', '未知')
    
    # 获取误差统计
    mae = error_stats.get('mean_absolute_error', 0)
    max_ae = error_stats.get('max_absolute_error', 0)
    med_ae = error_stats.get('median_absolute_error', 0)
    mape = error_stats.get('mean_relative_error', 0)
    
    # 准备报告内容
    report = f"""
=========================================
    PINN与蒙特卡洛方法对比分析报告
=========================================

分析时间: {time_module.strftime("%Y-%m-%d %H:%M:%S")}

模型参数:
---------
π = {pi_val}
β = {beta_val}
α = {alpha_val}
无风险利率 (r) = {r_val}
资产参数 (T) = {T_val}
到期时间 (τ) = {tau_val}
行权价 (K) = {K_val}

数据来源:
---------
PINN结果文件夹: {os.path.basename(pinn_dir)}
MC结果文件夹: {os.path.basename(mc_dir)}
对比结果文件夹: {os.path.basename(comp_dir)}

关键误差统计:
------------
平均绝对误差 (MAE) = {mae:.6f}
最大绝对误差 (Max AE) = {max_ae:.6f}
绝对误差中位数 (Median AE) = {med_ae:.6f}
平均相对误差 (MAPE) = {mape:.2f}%

分析与结论:
-----------
1. 总体精度:
   - 平均绝对误差为 {mae:.4f}，表明两种方法在大多数区域具有良好的一致性
   - 平均相对误差为 {mape:.2f}%，显示结果在百分比上接近

2. 局部区域差异:
   - 最大绝对误差为 {max_ae:.4f}，主要出现在期权到期时间 (t={tau_val}) 附近
   - 行权价 (K={K_val}) 附近区域也观察到较高误差

3. 实际应用建议:
   - PINN方法计算速度快，适合实时定价
   - MC方法结果稳定，适合精确计算
   - 对于大多数应用场景，两种方法可互换使用

报告生成工具: MC-PINN 对比分析系统
"""
    # 保存报告
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report)
    
    print(report)
    return report_path

# ================= 主函数 =================
def main():
    print("开始PINN与蒙特卡洛结果对比分析")
    
    # 创建比较结果保存目录
    timestamp = time_module.strftime("%Y%m%d-%H%M%S")
    comp_dir = os.path.join(RESULTS_ROOT, f"comparison_{timestamp}")
    os.makedirs(comp_dir, exist_ok=True)
    print(f"比较结果将保存至: {comp_dir}")
    
    # 查找结果文件夹
    try:
        pinn_dir, mc_dir = find_results_folders()
        if not pinn_dir or not mc_dir:
            print("错误: 无法找到足够的结果文件夹")
            print("请确保在以下路径中存在PINN和MC结果文件夹:")
            print(os.path.abspath(RESULTS_ROOT))
            return
    except Exception as e:
        print(f"查找结果文件夹时出错: {e}")
        return
    
    print(f"\n使用PINN文件夹: {pinn_dir}")
    print(f"使用MC文件夹: {mc_dir}")
    
    # 加载数据
    try:
        pinn_data, pinn_params = load_results(pinn_dir)
        mc_data, mc_params = load_results(mc_dir)
    except Exception as e:
        print(f"加载结果时出错: {e}")
        return
    
    # 合并参数
    params = {**pinn_params, **mc_params}
    print("\n合并的参数:")
    for k, v in params.items():
        print(f" - {k}: {v}")
    
    # 提取数据列
    try:
        if pinn_data.shape[1] < 3:
            raise ValueError(f"PINN数据维度不足: {pinn_data.shape}")
        if mc_data.shape[1] < 3:
            raise ValueError(f"MC数据维度不足: {mc_data.shape}")
        
        A_pinn = pinn_data[:, 0]
        t_pinn = pinn_data[:, 1]
        C_pinn = pinn_data[:, 2]
        
        A_mc = mc_data[:, 0]
        t_mc = mc_data[:, 1]
        C_mc = mc_data[:, 2]
    except Exception as e:
        print(f"提取数据出错: {e}")
        print(f"PINN数据形状: {pinn_data.shape if 'pinn_data' in locals() else 'N/A'}")
        print(f"MC数据形状: {mc_data.shape if 'mc_data' in locals() else 'N/A'}")
        return
    
    # 打印信息摘要
    print("\n数据分析摘要:")
    print(f"PINN数据点数量: {len(A_pinn)} (资产价格范围: {min(A_pinn):.2f}-{max(A_pinn):.2f})")
    print(f"MC数据点数量: {len(A_mc)} (资产价格范围: {min(A_mc):.2f}-{max(A_mc):.2f})")
    
    # 创建比较网格
    try:
        print("\n创建比较网格...")
        A_grid, t_grid, C_pinn_grid, C_mc_grid, abs_error, rel_error = create_comparison_grid(
            A_pinn, t_pinn, C_pinn, 
            A_mc, t_mc, C_mc
        )
    except Exception as e:
        print(f"创建比较网格时出错: {e}")
        return
    
    # 分析误差统计
    error_plot_path = os.path.join(comp_dir, "error_distribution.png")
    try:
        error_stats, _ = analyze_error_statistics(abs_error, rel_error, error_plot_path)
    except Exception as e:
        print(f"分析误差统计时出错: {e}")
        error_stats = {}
    
    # 绘制3D比较图
    comp_3d_path = os.path.join(comp_dir, "3d_comparison.png")
    try:
        K_val = params.get('k', 50)
        title = f"PINN与蒙特卡洛方法比较 (K={K_val})"
        plot_3d_comparison(A_grid, t_grid, C_pinn_grid, C_mc_grid, abs_error, title, comp_3d_path)
    except Exception as e:
        print(f"绘制3D图时出错: {e}")
    
    # 绘制时间序列比较
    time_series_path = os.path.join(comp_dir, "time_series_comparison.png")
    try:
        plot_time_series_comparison(A_pinn, t_pinn, C_pinn, A_mc, t_mc, C_mc, params, time_series_path)
    except Exception as e:
        print(f"绘制时间序列图时出错: {e}")
    
    # 绘制误差热图
    heatmaps_path = os.path.join(comp_dir, "error_heatmaps.png")
    try:
        plot_error_heatmaps(A_grid, t_grid, abs_error, rel_error, params, heatmaps_path)
    except Exception as e:
        print(f"绘制误差热图时出错: {e}")
    
    # 创建分析报告
    try:
        report_path = create_comparison_report(pinn_dir, mc_dir, comp_dir, params, error_stats)
    except Exception as e:
        print(f"创建报告时出错: {e}")
        report_path = os.path.join(comp_dir, "error_report.txt")
        with open(report_path, 'w') as f:
            f.write(f"报告生成失败: {e}")
    
    print("\n分析完成!所有结果保存至:")
    print(comp_dir)

if __name__ == "__main__":
    main()